In [17]:
import helpers as hp
import importlib, eda as de
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
importlib.reload(de) 
import os

In [2]:
x_train, x_test, y_train, train_ids, test_ids = hp.load_csv_data("dataset")

### Clean values such as "7/9/99" coded as non-answer in a survey

In [4]:
feature_names = np.genfromtxt("dataset/x_test.csv", delimiter=",", dtype=str, max_rows=1)

In [6]:
x_train_1 = de.replace_survey_codes_by_pattern(x_train, feature_names=feature_names)
x_test_1 = de.replace_survey_codes_by_pattern(x_test, feature_names=feature_names)

[CSTATE] → replaced values 7.0, 9.0 (2-code block) with NaN
[LANDLINE] → replaced values 77.0, 99.0 (2-code block) with NaN
[HHADULT] → replaced values 7.0, 9.0 (2-code block) with NaN
[GENHLTH] → replaced value 99.0 (single code) with NaN
[PHYSHLTH] → replaced value 99.0 (single code) with NaN
[MENTHLTH] → replaced value 99.0 (single code) with NaN
[POORHLTH] → replaced values 7.0, 9.0 (2-code block) with NaN
[HLTHPLN1] → replaced values 7.0, 9.0 (2-code block) with NaN
[PERSDOC2] → replaced values 7.0, 9.0 (2-code block) with NaN
[CHECKUP1] → replaced values 7.0, 9.0 (2-code block) with NaN
[BPHIGH4] → replaced values 7.0, 9.0 (2-code block) with NaN
[BPMEDS] → replaced values 7.0, 9.0 (2-code block) with NaN
[BLOODCHO] → replaced values 7.0, 9.0 (2-code block) with NaN
[CHOLCHK] → replaced values 7.0, 9.0 (2-code block) with NaN
[TOLDHI2] → replaced values 7.0, 9.0 (2-code block) with NaN
[CVDSTRK3] → replaced values 7.0, 9.0 (2-code block) with NaN
[ASTHMA3] → replaced values 7.0, 

### Split features into 3 categories : categorical, continuous, binary
### and removing ID/date features

In [11]:
# 1️⃣ Detect feature types on the clean dataset
categorical_train, continuous_train, binary_train = de.detect_feature_types_refined(x_train_1)

# 2️⃣ Detect date/ID-like features within continuous ones
suspect_features = de.detect_date_or_id_features(x_train_1[:, continuous_train])
removed_from_continuous = []

for idx in suspect_features:
    if idx in continuous_train:
        removed_from_continuous.append(idx)
        continuous_train.remove(idx)

print("Removed suspect date/ID-like features:", suspect_features)
print("Number removed:", len(removed_from_continuous))

# 3️⃣ Delete the suspect columns from both train and test
x_train_2 = np.delete(x_train_1, removed_from_continuous, axis=1)
x_test_2  = np.delete(x_test_1,  removed_from_continuous, axis=1)

# 4️⃣ Recalculate new feature type lists (now indices are coherent)
categorical_train, continuous_train, binary_train = de.detect_feature_types_refined(x_train_2)

print(f"Total categorical: {len(categorical_train)}, continuous: {len(continuous_train)}, binary: {len(binary_train)}")


Removed suspect date/ID-like features: [1, 3, 4, 5, 6, 31, 32, 41, 42, 55, 72, 73, 75, 76, 77, 78, 79, 80]
Number removed: 5
Total categorical: 101, continuous: 76, binary: 70


### Standardize continuous features

In [12]:
x_train_cont_std, x_test_cont_std = de.standardize(x_train_2[:, continuous_train], x_test_2[:, continuous_train])

print("Original mean (feature 0):", np.mean(x_train_2[:,0]))
print("Normalized mean (feature 0):", np.mean(x_train_cont_std[:,0]))

print("Original std (feature 0):", np.std(x_train_2[:,0]))
print("Normalized std (feature 0):", np.std(x_train_cont_std[:,0]))

Original mean (feature 0): 29.973651088728726
Normalized mean (feature 0): -9.843897410411615e-17
Original std (feature 0): 16.031752874847626
Normalized std (feature 0): 1.0


### One-hot encoding for categorical features

In [13]:
x_train_cat_encoded, cat_feature_names, skipped_features = de.one_hot_encode_numpy(
    x_train_2[:, categorical_train],
    feature_names=[feature_names[i] for i in categorical_train],
    max_categories=30  # can adjust regarding the dataset
)

print("Encoded shape:", x_train_cat_encoded.shape)
print("Encoded columns:", cat_feature_names)
print("Skipped features:", skipped_features)

Encoded shape: (328135, 509)
Encoded columns: ['_STATE=1', '_STATE=2', '_STATE=3', '_STATE=4', '_STATE=5', '_STATE=6', '_STATE=7', '_STATE=8', '_STATE=9', '_STATE=10', '_STATE=11', '_STATE=12', 'IDATE=1', 'IDATE=2', 'IDATE=3', 'IDATE=4', 'IDATE=5', 'IDATE=6', 'IDATE=7', 'IDATE=8', 'IDATE=9', 'IDATE=10', 'IDATE=11', 'IDATE=12', 'LADULT=0', 'LADULT=1', 'LADULT=2', 'LADULT=3', 'LADULT=4', 'LADULT=5', 'LADULT=6', 'LADULT=7', 'LADULT=8', 'LADULT=9', 'LADULT=10', 'LADULT=18', 'NUMADULT=0', 'NUMADULT=1', 'NUMADULT=2', 'NUMADULT=3', 'NUMADULT=4', 'NUMADULT=5', 'NUMADULT=6', 'NUMADULT=7', 'NUMADULT=8', 'NUMADULT=9', 'NUMADULT=10', 'LANDLINE=1', 'LANDLINE=2', 'LANDLINE=3', 'LANDLINE=4', 'LANDLINE=5', 'POORHLTH=1', 'POORHLTH=2', 'POORHLTH=3', 'PERSDOC2=1', 'PERSDOC2=2', 'PERSDOC2=3', 'PERSDOC2=4', 'PERSDOC2=7', 'PERSDOC2=8', 'PERSDOC2=9', 'MEDCOST=1', 'MEDCOST=2', 'MEDCOST=3', 'MEDCOST=4', 'BPMEDS=1', 'BPMEDS=2', 'BPMEDS=3', 'BPMEDS=4', 'ADDEPEV2=1', 'ADDEPEV2=2', 'ADDEPEV2=3', 'ADDEPEV2=4', 'DIA

### Concatenate everything to get our cleaned data

In [15]:
# ============================================================
# 🔹 FINAL PREPROCESSING PIPELINE
# ============================================================

# --- 1️⃣ Continuous features: Standardize (train stats only)
x_train_cont_final, x_test_cont_final = de.standardize(
    x_train_2[:, continuous_train],
    x_test_2[:, continuous_train]
)

print("✅ Continuous features standardized.")
print("Train continuous shape:", x_train_cont_final.shape)
print("Test continuous shape :", x_test_cont_final.shape)


# --- 2️⃣ Binary features: Keep as-is
x_train_bin = x_train_2[:, binary_train]
x_test_bin  = x_test_2[:, binary_train]

print("✅ Binary features selected.")
print("Binary shape (train):", x_train_bin.shape)


# --- 3️⃣ Categorical features: One-hot encode (train)
x_train_cat_encoded, cat_feature_names, skipped_features = de.one_hot_encode_numpy(
    x_train_2[:, categorical_train],
    feature_names=[feature_names[i] for i in categorical_train],
    max_categories=30
)

print("✅ Train categorical one-hot encoded.")
print("Train categorical shape:", x_train_cat_encoded.shape)
print("Skipped features:", skipped_features)


# --- 4️⃣ Encode test categorical features with the same categories as the train
unique_per_feature = [
    np.unique(x_train_2[:, j][~np.isnan(x_train_2[:, j])])
    for j in categorical_train
]

encoded_blocks_test = []
for j, uniques in enumerate(unique_per_feature):
    col = x_test_2[:, categorical_train[j]]
    encoded = np.zeros((col.shape[0], len(uniques)))
    for i, val in enumerate(uniques):
        mask = (col == val)
        encoded[mask, i] = 1.0
    encoded_blocks_test.append(encoded)

x_test_cat_encoded = np.concatenate(encoded_blocks_test, axis=1)

print("✅ Test categorical one-hot encoded (using train categories).")
print("Test categorical shape:", x_test_cat_encoded.shape)


# --- 5️⃣ Concatenate all blocks
x_train_final = np.concatenate([x_train_cont_final, x_train_cat_encoded, x_train_bin], axis=1)
x_test_final  = np.concatenate([x_test_cont_final, x_test_cat_encoded, x_test_bin], axis=1)

print("============================================================")
print("✅ Final preprocessing complete.")
print("Final training shape:", x_train_final.shape)
print("Final testing shape :", x_test_final.shape)
print("============================================================")


✅ Continuous features standardized.
Train continuous shape: (328135, 76)
Test continuous shape : (109379, 76)
✅ Binary features selected.
Binary shape (train): (328135, 70)
✅ Train categorical one-hot encoded.
Train categorical shape: (328135, 509)
Skipped features: []
✅ Test categorical one-hot encoded (using train categories).
Test categorical shape: (109379, 509)
✅ Final preprocessing complete.
Final training shape: (328135, 655)
Final testing shape : (109379, 655)


In [18]:
os.makedirs("dataset_clean", exist_ok=True)

np.savetxt("dataset_clean/x_train.csv", x_train_final, delimiter=",")
np.savetxt("dataset_clean/x_test.csv", x_test_final, delimiter=",")
np.savetxt("dataset_clean/y_train.csv", y_train, delimiter=",")